# BTYD: BG/NBD + Gamma-Gamma LTV Model

Классический вероятностный подход к предсказанию LTV, явно рекомендованный организаторами.

## Идея
- **BG/NBD** (Beta-Geometric / Negative Binomial Distribution): моделирует частоту покупок и P(alive) — вероятность, что пользователь ещё активен
- **Gamma-Gamma**: моделирует средний чек (E[GMV | purchase])
- **Итог:** E[GMV за 30 дней] = E[N покупок] × E[GMV за покупку]

## Входные данные для BTYD
- `frequency` — число повторных покупок (total_orders - 1)
- `recency` — время от первой до последней покупки (в неделях)
- `T` — время с момента первой покупки до anchor date (в неделях)
- `monetary_value` — средний GMV за одну покупку

In [ ]:
import subprocess, sys
try:
    import lifetimes
    print(f'lifetimes {lifetimes.__version__} уже установлен')
except ImportError:
    print('Устанавливаю lifetimes...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lifetimes', '-q'])
    import lifetimes
    print(f'Установлено: lifetimes {lifetimes.__version__}')

lifetimes 0.11.3 уже установлен


In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date, timedelta
from sklearn.metrics import mean_squared_error

from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')

ANCHOR_DATE = date(2026, 2, 13)
TARGET_START = date(2026, 2, 14)
TARGET_END   = date(2026, 3, 15)
TARGET_DAYS  = (TARGET_END - TARGET_START).days + 1
TARGET_WEEKS = TARGET_DAYS / 7

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

print(f'Горизонт предсказания: {TARGET_DAYS} дней ({TARGET_WEEKS:.2f} недель)')

Горизонт предсказания: 30 дней (4.29 недель)


## Загрузка и агрегация данных

Нам нужны транзакционные данные: даты покупок и суммы для каждого пользователя.

In [3]:
print('Загружаем данные...')
df = pl.read_parquet(DATA_DIR / 'train.parquet')
print(f'Размер: {df.shape}')
print(f'Колонки: {df.columns}')
print(f'Период: {df["event_date"].min()} -- {df["event_date"].max()}')

Загружаем данные...
Размер: (30631006, 18)
Колонки: ['event_date', 'user_id', 'search', 'cat', 'has_search_to_cart', 'has_search_to_ord', 'has_cat_to_cart', 'has_cat_to_ord', 'search_to_cart', 'search_to_ord', 'cat_to_cart', 'cat_to_ord', 'gmv_search', 'gmv_cat', 'to_cart', 'to_ord', 'gmv', 'searches']
Период: 2025-01-01 -- 2026-02-13


In [ ]:
purchases = df.filter(pl.col('gmv') > 0).select(['user_id', 'event_date', 'gmv'])
purchases_pd = purchases.to_pandas()
purchases_pd['event_date'] = pd.to_datetime(purchases_pd['event_date'])

anchor_dt = pd.Timestamp(ANCHOR_DATE)

print(f'Всего записей с gmv>0: {len(purchases_pd):,}')
print(f'Уникальных покупателей: {purchases_pd["user_id"].nunique():,}')

all_users = df.select('user_id').unique().to_pandas()
print(f'Всего пользователей: {len(all_users):,}')

Всего записей с gmv>0: 4,736,907
Уникальных покупателей: 219,168
Всего пользователей: 250,000


In [ ]:
print('Строим RFM-таблицу...')

rfm = summary_data_from_transaction_data(
    purchases_pd,
    customer_id_col='user_id',
    datetime_col='event_date',
    monetary_value_col='gmv',
    observation_period_end=anchor_dt,
    freq='W'
)

print(f'RFM таблица: {rfm.shape}')
print(rfm.describe())
print(f'\nПользователи с frequency=0: {(rfm["frequency"]==0).sum():,} (разовые покупки)')  

Строим RFM-таблицу...
RFM таблица: (219168, 4)
          frequency        recency              T  monetary_value
count  219168.00000  219168.000000  219168.000000   219168.000000
mean       13.53756      38.320024      44.429848       65.128100
std        12.58761      19.071386      15.606374       91.066247
min         0.00000       0.000000       0.000000        0.000000
25%         3.00000      25.000000      36.000000       23.368399
50%        10.00000      46.000000      52.000000       44.506074
75%        21.00000      54.000000      56.000000       79.327695
max        58.00000      58.000000      58.000000     6372.424163

Пользователи с frequency=0: 17,090 (разовые покупки)


In [ ]:
rfm_returning = rfm[rfm['frequency'] > 0]
print(f'Пользователи с повторными покупками: {len(rfm_returning):,}')
print(f'Их доля: {len(rfm_returning)/len(rfm)*100:.1f}%')

print(f'\nСтатистика frequency (повторных покупок):')
print(f'  mean={rfm["frequency"].mean():.2f}  median={rfm["frequency"].median():.2f}  max={rfm["frequency"].max()}')
print(f'\nСтатистика monetary_value:')
print(f'  mean={rfm_returning["monetary_value"].mean():.2f}  median={rfm_returning["monetary_value"].median():.2f}')

Пользователи с повторными покупками: 202,078
Их доля: 92.2%

Статистика frequency (повторных покупок):
  mean=13.54  median=10.00  max=58.0

Статистика monetary_value:
  mean=70.64  median=48.55


## Stage 1: BG/NBD — моделирование частоты покупок

In [7]:
print('Обучаем BG/NBD модель...')
bgf = BetaGeoFitter(penalizer_coef=0.001)
bgf.fit(
    frequency=rfm['frequency'],
    recency=rfm['recency'],
    T=rfm['T'],
    verbose=True
)
print('\nПараметры BG/NBD:')
print(bgf.params_)

Обучаем BG/NBD модель...
Optimization terminated successfully.
         Current function value: -27.603328
         Iterations: 33
         Function evaluations: 36
         Gradient evaluations: 36

Параметры BG/NBD:
r        1.658183
alpha    5.812250
a        0.002080
b        0.460693
dtype: float64


In [ ]:
rfm['predicted_purchases'] = bgf.conditional_expected_number_of_purchases_up_to_time(
    t=TARGET_WEEKS,
    frequency=rfm['frequency'],
    recency=rfm['recency'],
    T=rfm['T']
)

rfm['prob_alive'] = bgf.conditional_probability_alive(
    frequency=rfm['frequency'],
    recency=rfm['recency'],
    T=rfm['T']
)

print('BG/NBD предсказания:')
print(f'  Ожид. покупок за {TARGET_DAYS}д:')
print(f'    mean={rfm["predicted_purchases"].mean():.4f}')
print(f'    median={rfm["predicted_purchases"].median():.4f}')
print(f'  P(alive): mean={rfm["prob_alive"].mean():.4f}')

BG/NBD предсказания:
  Ожид. покупок за 30д:
    mean=1.2191
    median=1.0232
  P(alive): mean=0.9916


## Stage 2: Gamma-Gamma — моделирование среднего чека

In [ ]:
print('Обучаем Gamma-Gamma модель...')
ggf = GammaGammaFitter(penalizer_coef=0.001)
ggf.fit(
    frequency=rfm_returning['frequency'],
    monetary_value=rfm_returning['monetary_value'],
    verbose=True
)
print('\nПараметры Gamma-Gamma:')
print(ggf.params_)

Обучаем Gamma-Gamma модель...
Optimization terminated successfully.
         Current function value: 5.274652
         Iterations: 16
         Function evaluations: 17
         Gradient evaluations: 17

Параметры Gamma-Gamma:
p    6.347325
q    1.402675
v    7.418067
dtype: float64


In [ ]:
rfm_returning_pred = rfm[rfm['frequency'] > 0].copy()
rfm_returning_pred['expected_avg_gmv'] = ggf.conditional_expected_average_profit(
    frequency=rfm_returning_pred['frequency'],
    monetary_value=rfm_returning_pred['monetary_value']
)

rfm_one_time = rfm[rfm['frequency'] == 0].copy()
rfm_one_time['expected_avg_gmv'] = rfm_one_time['monetary_value']

rfm_all_gmv = pd.concat([rfm_returning_pred, rfm_one_time])

print('Gamma-Gamma предсказания (средний чек):')
print(f'  mean={rfm_returning_pred["expected_avg_gmv"].mean():.2f}')
print(f'  median={rfm_returning_pred["expected_avg_gmv"].median():.2f}')

Gamma-Gamma предсказания (средний чек):
  mean=71.34
  median=49.17


## Финальное предсказание LTV

$$\hat{y} = E[N_{\text{покупок}}] \times E[\text{GMV за покупку}]$$

In [ ]:
predictions = rfm[['predicted_purchases', 'prob_alive']].copy()
predictions = predictions.join(rfm_all_gmv[['expected_avg_gmv']], how='left')
predictions['expected_avg_gmv'] = predictions['expected_avg_gmv'].fillna(0)

predictions['ltv_btyd'] = (
    predictions['predicted_purchases'] * predictions['expected_avg_gmv']
).clip(lower=0)

all_users_df = all_users.set_index('user_id')
final = all_users_df.join(predictions[['ltv_btyd']], how='left')
final['ltv_btyd'] = final['ltv_btyd'].fillna(0)
final = final.reset_index()

print('Финальные BTYD предсказания:')
print(f'  Пользователей: {len(final):,}')
print(f'  Нулей: {(final["ltv_btyd"]==0).sum():,} ({(final["ltv_btyd"]==0).mean()*100:.1f}%)')
print(f'  mean={final["ltv_btyd"].mean():.2f}  median={final["ltv_btyd"].median():.2f}')

submit = final[['user_id', 'ltv_btyd']].rename(columns={'ltv_btyd': 'predict'})
submit.to_csv(PROCESSED_DIR / 'btyd_submission.csv', index=False)
print(f'\nСохранено: btyd_submission.csv')

Финальные BTYD предсказания:
  Пользователей: 250,000
  Нулей: 47,922 (19.2%)
  mean=85.17  median=36.33

Сохранено: btyd_submission.csv


## Блендинг BTYD с ML моделями

BTYD и CatBoost моделируют принципиально разные аспекты:
- BTYD: вероятностная модель покупательского поведения
- CatBoost: нелинейная функция от агрегированных признаков

Их комбинация потенциально ортогональна.

In [ ]:
import scipy.stats as stats

cb_df  = pd.read_csv(PROCESSED_DIR / 'catboost_v3_submission.csv').sort_values('user_id').reset_index(drop=True)
rnn_df = pd.read_csv(PROCESSED_DIR / 'rnn_v3_submission.csv').sort_values('user_id').reset_index(drop=True)

btyd_sorted = final.sort_values('user_id').reset_index(drop=True)

assert list(cb_df['user_id']) == list(btyd_sorted['user_id']), 'user_id mismatch!'

p_btyd = btyd_sorted['ltv_btyd'].values
p_cb   = cb_df['predict'].values
p_rnn  = rnn_df['predict'].values

print('Корреляции BTYD с ML моделями:')
print(f'  BTYD vs CB : Pearson={np.corrcoef(p_btyd, p_cb)[0,1]:.4f}  Spearman={stats.spearmanr(p_btyd, p_cb).statistic:.4f}')
print(f'  BTYD vs RNN: Pearson={np.corrcoef(p_btyd, p_rnn)[0,1]:.4f}  Spearman={stats.spearmanr(p_btyd, p_rnn).statistic:.4f}')

print('\nГенерация блендов BTYD + CB v3:')
for alpha in [0.2, 0.3, 0.4, 0.5]:
    blend = (alpha * p_btyd + (1 - alpha) * p_cb).clip(min=0)
    fname = f'btyd_blend_btyd{int(alpha*10)}_cb{int((1-alpha)*10)}.csv'
    pd.DataFrame({'user_id': cb_df['user_id'], 'predict': blend}).to_csv(
        PROCESSED_DIR / fname, index=False)
    print(f'  BTYD {int(alpha*100)}% + CB {int((1-alpha)*100)}%: mean={blend.mean():.2f} → {fname}')

print('\nГенерация блендов BTYD + CB + RNN:')
for w_btyd, w_cb, w_rnn in [(0.2, 0.5, 0.3), (0.15, 0.55, 0.30), (0.25, 0.5, 0.25)]:
    blend = (w_btyd * p_btyd + w_cb * p_cb + w_rnn * p_rnn).clip(min=0)
    fname = f'btyd3_b{int(w_btyd*100)}_cb{int(w_cb*100)}_rnn{int(w_rnn*100)}.csv'
    pd.DataFrame({'user_id': cb_df['user_id'], 'predict': blend}).to_csv(
        PROCESSED_DIR / fname, index=False)
    print(f'  BTYD {int(w_btyd*100)}%+CB {int(w_cb*100)}%+RNN {int(w_rnn*100)}%: mean={blend.mean():.2f} → {fname}')

print('\nГотово!')

Корреляции BTYD с ML моделями:
  BTYD vs CB : Pearson=0.7578  Spearman=0.8740
  BTYD vs RNN: Pearson=0.7155  Spearman=0.8572

Генерация блендов BTYD + CB v3:
  BTYD 20% + CB 80%: mean=51.33 → btyd_blend_btyd2_cb8.csv
  BTYD 30% + CB 70%: mean=55.56 → btyd_blend_btyd3_cb7.csv
  BTYD 40% + CB 60%: mean=59.79 → btyd_blend_btyd4_cb6.csv
  BTYD 50% + CB 50%: mean=64.02 → btyd_blend_btyd5_cb5.csv

Генерация блендов BTYD + CB + RNN:
  BTYD 20%+CB 50%+RNN 30%: mean=51.13 → btyd3_b20_cb50_rnn30.csv
  BTYD 15%+CB 55%+RNN 30%: mean=49.01 → btyd3_b15_cb55_rnn30.csv
  BTYD 25%+CB 50%+RNN 25%: mean=53.27 → btyd3_b25_cb50_rnn25.csv

Готово!
